
## Milestone 2: Core Challenge - CNN Optimization and Diagnosis 


######  Design and Train Baseline Model: Design and implement a fully custom Classification CNN (e.g., ≥4 layers). (2 points)

###### This CNN must output a binary prediction: "Image contains Target Object" vs. "Image does NOT contain Target Object."

###### Train a Baseline Model for a fixed number of epochs without any regularization.

##### splitting the zip files containing images to training,validation and testing datasets and saving in folders

In [13]:
import os
import zipfile
import shutil
import random

# Paths
base_dir = "dataset"

os.makedirs(base_dir, exist_ok=True)

# Your ZIP files
zip_files = {
    "wine_glass": "New_Folder/wine-glass.zip",
    "not_wine_glass": "New_Folder/not-wine-glass.zip"
}

#  Unzip each folder
for label, zip_path in zip_files.items():
    extract_path = os.path.join(base_dir, label)
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    subdirs = [d for d in os.listdir(extract_path) if os.path.isdir(os.path.join(extract_path, d))]
    if len(subdirs) == 1:
        nested_dir = os.path.join(extract_path, subdirs[0])
        for item in os.listdir(nested_dir):
            shutil.move(os.path.join(nested_dir, item), extract_path)
        shutil.rmtree(nested_dir)

# Create split directories
splits = ['train', 'val', 'test']
for split in splits:
    for label in zip_files.keys():
        os.makedirs(os.path.join(base_dir, split, label), exist_ok=True)

# Split data
split_ratios = {'train': 0.8, 'val': 0.15, 'test': 0.05}

for label in zip_files.keys():
    source_dir = os.path.join(base_dir, label)
    images = os.listdir(source_dir)
    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * split_ratios['train'])
    n_val = int(n_total * split_ratios['val'])

    splits_data = {
        'train': images[:n_train],
        'val': images[n_train:n_train + n_val],
        'test': images[n_train + n_val:]
    }

    for split, split_images in splits_data.items():
        for img in split_images:
            src = os.path.join(source_dir, img)
            dst = os.path.join(base_dir, split, label, img)
            shutil.move(src, dst)

    


In [14]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'  # binary classification
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

Found 160 files belonging to 2 classes.
Found 30 files belonging to 2 classes.
Found 10 files belonging to 2 classes.


In [3]:
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))


In [4]:
model = models.Sequential([
    layers.Conv2D(16, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # binary output
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10  
)
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]

print(f"Training Accuracy: {train_acc*100:.2f}%")
print(f"Validation Accuracy: {val_acc*100:.2f}%")


Epoch 1/10


C:\Users\Methruchi Peiris\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 217ms/step - accuracy: 0.8875 - loss: 0.4526 - val_accuracy: 0.9000 - val_loss: 0.3037
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step - accuracy: 0.9000 - loss: 0.4256 - val_accuracy: 0.9000 - val_loss: 0.3659
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 155ms/step - accuracy: 0.9000 - loss: 0.3366 - val_accuracy: 0.9000 - val_loss: 0.3413
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 146ms/step - accuracy: 0.9000 - loss: 0.3422 - val_accuracy: 0.9000 - val_loss: 0.3053
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 149ms/step - accuracy: 0.9000 - loss: 0.3171 - val_accuracy: 0.9000 - val_loss: 0.3064
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 151ms/step - accuracy: 0.9000 - loss: 0.3326 - val_accuracy: 0.9000 - val_loss: 0.3076
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 146ms/step - accuracy: 0.9000 - loss: 0.3066 - val_accuracy: 0.9000 - val_loss: 0.3026
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 141ms/step - accuracy: 0.9000 - loss: 0.2998 - val_accuracy: 0.9000 - val_loss: 0.3008
Epo

In [4]:
## 2. Bias-Variance Diagnosis: Analyze the Baseline Model's performance and determine the primary problem using the following metrics: (2 points)

### Training Accuracy: 90.00%
### Validation Accuracy: 90.00% 

### This scenario indicates a high bias but no variance observed at this point

#### 3. Systematic Optimization and Regularization (The Recipe): Based on your diagnosis, apply the necessary techniques learned in the course to solve the problem: (2 points)

#### · If High Bias: Increase Network Complexity (more layers/filters) or train longer (managing with Callbacks).

#### · If High Variance: Apply L2 Regularization to the final dense layers AND/OR implement Dropout layers.

### Since the human error could be considered as zero, we can see that there is a high bias of 10% hence we increase layers and make the architecture complex.Also epochs are increased from 10 to 50

In [5]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # binary output
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50 
)
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]

print(f"Training Accuracy: {train_acc*100:.2f}%")
print(f"Validation Accuracy: {val_acc*100:.2f}%")


Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 372ms/step - accuracy: 0.7375 - loss: 0.4646 - val_accuracy: 0.9000 - val_loss: 0.3438
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 280ms/step - accuracy: 0.9000 - loss: 0.4383 - val_accuracy: 0.9000 - val_loss: 0.3073
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 279ms/step - accuracy: 0.9000 - loss: 0.3883 - val_accuracy: 0.9000 - val_loss: 0.3177
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 279ms/step - accuracy: 0.9000 - loss: 0.3232 - val_accuracy: 0.9000 - val_loss: 0.3346
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 281ms/step - accuracy: 0.9000 - loss: 0.3326 - val_accuracy: 0.9000 - val_loss: 0.3218
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 280ms/step - accuracy: 0.9000 - loss: 0.3124 - val_accuracy: 0.9000 - val_loss: 0.3105
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 280ms/step - accuracy: 0.9000 - loss: 0.3084 - val_accuracy: 0.9000 - val_loss: 0.3241
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 262ms/step - accuracy: 0.9000 - loss: 0.3013 - val_accuracy: 0.9000 - val_loss:

### The results indicated a training accuracy of 100% and a validation accuracy of 93.3% creating a variance. However, in this stage, we are just adding callbacks with a patience of 8

In [6]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping_callback = EarlyStopping(
    monitor='val_loss', # The metric to monitor for improvement
    patience=8,         # Number of epochs with no improvement after which training will be stopped
    verbose=1,          # Show a message when stopping
    restore_best_weights=True # Restores model weights from the epoch with the best value of the monitored quantity.
)

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # binary output
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

all_callbacks = [early_stopping_callback]
history = model.fit(
    train_ds,
    validation_data=val_ds,
    callbacks=all_callbacks,
    epochs=50 
)
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]

print(f"Training Accuracy: {train_acc*100:.2f}%")
print(f"Validation Accuracy: {val_acc*100:.2f}%")


Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 325ms/step - accuracy: 0.7563 - loss: 0.6077 - val_accuracy: 0.9000 - val_loss: 0.3032
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 285ms/step - accuracy: 0.9000 - loss: 0.4366 - val_accuracy: 0.9000 - val_loss: 0.4278
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 272ms/step - accuracy: 0.9000 - loss: 0.3704 - val_accuracy: 0.9000 - val_loss: 0.3188
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 282ms/step - accuracy: 0.9000 - loss: 0.3630 - val_accuracy: 0.9000 - val_loss: 0.3018
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 284ms/step - accuracy: 0.9000 - loss: 0.3265 - val_accuracy: 0.9000 - val_loss: 0.3244
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 284ms/step - accuracy: 0.9000 - loss: 0.3343 - val_accuracy: 0.9000 - val_loss: 0.2996
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 288ms/step - accuracy: 0.9000 - loss: 0.3440 - val_accuracy: 0.9000 - val_loss: 0.3065
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 269ms/step - accuracy: 0.9000 - loss: 0.3348 - val_accuracy: 0.9000 - val_loss:

### However with the callbacks of early stopping the training accuracy decreased to 90% creating a high bias with no variance

### In order to address the variance created without call backs instance, we are experimenting L2 and dropout to overcome the high variance. Here we included a slow learning rate of 1e-4 for stable learning and results 

In [7]:
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam


early_stopping_callback = EarlyStopping(
    monitor='val_loss', # The metric to monitor for improvement
    patience=8,         # Number of epochs with no improvement after which training will be stopped
    verbose=1,          # Show a message when stopping
    restore_best_weights=True # Restores model weights from the epoch with the best value of the monitored quantity.
)

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', 
                  kernel_regularizer=regularizers.l2(0.001),  # L2 regularization
                  input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
                  #kernel_regularizer=regularizers.l2(0.001)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
                  #kernel_regularizer=regularizers.l2(0.001)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
                  #kernel_regularizer=regularizers.l2(0.001)),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    #layers.Dropout(0.5),  # Dropout layer with 50% drop rate
    layers.Dense(1, activation='sigmoid')  # binary output
])

optimizer = Adam(learning_rate=1e-4)  # smaller learning rate

# Compile the model
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

all_callbacks = [early_stopping_callback]
history = model.fit(
    train_ds,
    validation_data=val_ds,
    callbacks=all_callbacks,
    epochs=100 
)
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]

print(f"Training Accuracy: {train_acc*100:.2f}%")
print(f"Validation Accuracy: {val_acc*100:.2f}%")


Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 306ms/step - accuracy: 0.9000 - loss: 0.6586 - val_accuracy: 0.9000 - val_loss: 0.5430
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 250ms/step - accuracy: 0.9000 - loss: 0.5049 - val_accuracy: 0.9000 - val_loss: 0.4124
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 234ms/step - accuracy: 0.9000 - loss: 0.4080 - val_accuracy: 0.9000 - val_loss: 0.3664
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 233ms/step - accuracy: 0.9000 - loss: 0.3970 - val_accuracy: 0.9000 - val_loss: 0.3767
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 236ms/step - accuracy: 0.9000 - loss: 0.4026 - val_accuracy: 0.9000 - val_loss: 0.3749
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 230ms/step - accuracy: 0.9000 - loss: 0.3944 - val_accuracy: 0.9000 - val_loss: 0.3649
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 233ms/step - accuracy: 0.9000 - loss: 0.3840 - val_accuracy: 0.9000 - val_loss: 0.3635
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 225ms/step - accuracy: 0.9000 - loss: 0.3840 - val_accuracy: 0.9000 - v

### The final Training Accuracy is 99.37% and the Validation Accuracy is 90.00%. A variance has been created. However despite many regularizations used, the validation accuracy did not improve. This might be due to low amount of data.

In [8]:
## Concatenate all label arrays from the test dataset into a single 1D array
import numpy as np

y_true = np.concatenate([labels for images, labels in test_ds], axis=0)


In [9]:
# Get the model's predicted probabilities on the test dataset
#Convert probabilities to binary class predictions (0 or 1) using 0.5 threshold
# and flatten the array to a 1D vector

y_prob = model.predict(test_ds)
y_pred = (y_prob > 0.5).astype("int32").flatten()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step


In [10]:
# calculate evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)


Accuracy: 1.0
Precision: 1.0
Recall: 1.0


In [11]:
# save the model weights
model.save("wineglass_classifier.h5")
